# Check start/end points head with coarse Naches ATS simulation

- **check the start/end points head for all hillslopes identified larger than 100m**
- generate BC head figures, part of the site selection within 900m grid identified
- based on the Naches-2 run between 2011 and 2014

Head extracted from Zhi's 3D ATS simulation for Naches

- file `global/cfs/cdirs/m1800/naches_run2_share/Naches-2`
- Information
    - "Time": from 11224 to 16059; unit is day;
        - 11400 = 85+365x31 --> 2011.03.26
        - 12631 = 221+365x34 --> 2014.08.09

Output of this script

- constant head at the starting and end points -> to drive the run0
- head at a typical year at the starting and end points -> to drive the run1
- transient head

In [ ]:
%load_ext autoreload
%autoreload 2

## config Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year_spinup         = config['start_year_spinup']
end_year_spinup           = config['end_year_spinup']
nyears_steadystate_spinup = config['nyears_steadystate_spinup']
nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
start_year_transient      = config['start_year_transient']
end_year_transient        = config['end_year_transient']

In [ ]:
outputs={}

## extract point raw data from 3D ATS simulation

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import h5py as h5

import scipy.signal
from datetime import datetime, timedelta
import pandas as pd

import ats_xdmf as xdmf
import time
import random
import pandas
import os

from scipy.io import loadmat
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping
import geopandas as gpd

In [ ]:
## get stream network
# Essential imports for watershed workflow
import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.plot
import pyproj

# set up a dictionary of source objects
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.source_list.log_sources(sources)

# Set up watershed workflow CRS (DayMet CRS)
crs_daymet = watershed_workflow.crs.daymet_crs()

# Set up sources
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']

# Parameters for river extraction
hucs_config = [config['hucs']]
ignore_small_rivers = 2
prune_by_area_fraction = 0.0

# Get HUC12 list
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
    return huc12_list

hucs = get_huc12(hucs_config)
huc_level = 12

print(f"Processing HUCs: {hucs[:5]}...")

# Load watershed HUCs
my_hucs = []
for huc in hucs:
    _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs_daymet)
    my_hucs.extend(ws)

watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

# Download/collect the river network
print("Downloading river network...")
_, reaches = watershed_workflow.get_reaches(sources['hydrography'], hucs[0], 
                                            watershed.exterior(), crs_daymet, crs_daymet,
                                            in_network=True, properties=True)

# Construct river network
rivers = watershed_workflow.construct_rivers(reaches, method='hydroseq',
                                             ignore_small_rivers=ignore_small_rivers,
                                             prune_by_area=prune_by_area_fraction * watershed.exterior().area * 1.e-6,
                                             remove_diversions=True,
                                             remove_braided_divergences=True)

print(f"Number of rivers: {len(rivers)}")

### Config 3D ATS-flow results

In [ ]:
# Define the output model directory, whre ats_vis data (.h5) are located
model_dir = '/global/cfs/cdirs/m1800/naches_run2_share/Naches-2'

# Define the Parameters AND verify with the XML
rho = 997 # density of water, kg m^-3
g = 9.80665 # gravity, m s^-2
patm = 101325 # atmopsheric pressure, Pascals

# Define raw output, and skip raw point data extraction if detect it
outputs['tmp_BChead_raw'] = f'../data-processed/{site_name}/tmp_bc_startend_raw.naches-2.h5'

# Check if file exists
if os.path.exists(outputs['tmp_BChead_raw']):
    flag_atsptextract = False
    print(f"File {outputs['tmp_BChead_raw']} exists, skipping extraction.")
else:
    flag_atsptextract = True
    print(f"File {outputs['tmp_BChead_raw']} not found, will extract data.")

flag_atsptextract = True

### Load all hillslopes larger than 100m

- These hillslopes have
    - end point in stream (or meet certain intermittency in the future)
    - pre-selected burn severity and LAI changes from MODIS
    - enough length identified through subcatchments
- however,
    - the boundary head of start points might still not be appropriate from the view of a coarse ATS model. so I want to plot the start/end boundary head, to further select an appropriate hillslope model to simulate further.

In [ ]:
# Load all hillslopes from the CSV file created by 0a-transect_latlon.Naches.D8.ipynb
csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_all_gt100m_summary.csv'
df = pd.read_csv(csv_filename)

# Extract coordinates - note the correct column names from the CSV
start_coords_all = df[['start_x', 'start_y']].values
end_coords_all = df[['end_x', 'end_y']].values

# Extract metadata
hillslope_names = df['transect_name'].tolist()
point_ids = df['point_id'].values
hillslope_ids = df['hillslope_id'].values
distances = df['distance_m'].values

num_transects = len(df)

print(f"Loaded {num_transects} hillslopes from: {csv_filename}")
print(f"\nHillslope information:")
print(f"{'Index':<8} {'Name':<12} {'Point ID':<10} {'Hillslope ID':<13} {'Distance (m)':<15}")
print("-" * 60)
for i in range(num_transects):
    print(f"{i:<8} {hillslope_names[i]:<12} {point_ids[i]:<10} {hillslope_ids[i]:<13} {distances[i]:<15.1f}")

print(f"\nStart coordinates shape: {start_coords_all.shape}")
print(f"End coordinates shape: {end_coords_all.shape}")

### Load surface/subsurface h5 files

In [ ]:
## a function to estimate WTD based on pressure

def get_ats_wtd_pressurebased(pressure_subsurface,visfile_surface, visfile_subsurface):
    # visfile_subsurface.centroids.shape -> (n_surface, 14, 3); 14 is soil layers, bottom-top
    iz_coord = visfile_subsurface.centroids[:,:,-1]
    
    ### Find Equivalent Surface and Subsurface ID based on cell centroids
    # take some care on the rounding of coordinates, can cause errors
    surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
    subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
    # [remarks] prepare to perform a pairwise compare [n_surface,1,2] with [1,n_surface,2] -> returns [n_surface, n_surface]
    surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
    subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
    matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
    surface_indices, subsurface_indices = np.nonzero(matches)
    surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))
    assert surface_subsurface_IDs[:,1].shape == visfile_surface.centroids[:,1].shape, f"Shape mismatch: change the round precision in the surface/subsurface centroid coordinates"
    
    ### Estimate WTD based on pressure
    # pressure head
    ih = (pressure_subsurface - patm) / (rho * g) # dim=(time, n_surface, 14)
    mask = ih > 0
    first_false_idx = (~mask).argmax(axis=-1) # dim=(time, n_surface)
    max_len = mask.shape[-1]
    indices = np.arange(max_len)
    mask2 = indices < first_false_idx[..., np.newaxis]
    all_true_rows = (first_false_idx == 0)
    mask2[all_true_rows] = True
    sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
    sat_idx = mask2.shape[-1] - 1 - sat_idx
    sat_idx[~mask2.any(axis=-1)] = 0
    # WTD elevation
    # [remark] iH_rev=part1+part2; part1 is the relative distance between water table and the first saturated subsurface cell
    iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx] 
    first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
    # WTD (from surface)
    head_pressure_based = -(iz_coord[:,-1]+first_depth_to_centroid - iH_rev) #[added by Yi]
    wtdep_pressure_based = np.maximum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    pwdep_pressure_based = - np.minimum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    assert pressure_subsurface[:,:,1].shape == wtdep_pressure_based.shape, f"Shape mismatch: Error in WTD calculation"
    
    ### Re-arrange WTD in accordance to surface IDs
    # As pressure obtained from subsurface does not follow surface ID, re-arrange this:
    head_rearranged  = head_pressure_based[:, subsurface_indices]
    wtdep_rearranged = wtdep_pressure_based[:, subsurface_indices]
    pwdep_rearranged = pwdep_pressure_based[:, subsurface_indices]
    
    return surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged

In [ ]:
if flag_atsptextract:
    # surface
    start = time.time()
    visfile_surface = xdmf.VisFile(directory=model_dir,
                                   domain="surface", 
                                   filename="ats_vis_surface_data.h5" , 
                                   mesh_filename="ats_vis_surface_mesh.h5")
    visfile_surface.loadMesh()
    end = time.time()
    print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
    
    print(visfile_surface.times)

    tmp_year = 2026 # a standard year
    day_number0 = int(visfile_surface.times[0])
    date0 = datetime.strptime(f'{tmp_year}-{day_number0%365}', '%Y-%j')
    print(f"t0: {day_number0}={day_number0%365}+365x{day_number0//365}")
    print(f"t0: {day_number0%365} is {date0.strftime('%B %d')}")
    
    day_number1 = int(visfile_surface.times[-1])
    date1 = datetime.strptime(f'{tmp_year}-{day_number1%365}', '%Y-%j')
    print(f"t1: {day_number1}={day_number1%365}+365x{day_number1//365}")
    print(f"t1: {day_number1%365} is {date1.strftime('%B %d')}")
    
else:
    print("Skipping: data already extracted")

In [ ]:
if flag_atsptextract:
    # subsurface
    start = time.time()
    visfile_subsurface = xdmf.VisFile(directory=model_dir,
                                      filename="ats_vis_data.h5", 
                                      mesh_filename="ats_vis_mesh.h5")
    visfile_subsurface.loadMesh(columnar=True)
    end = time.time()
    print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")
    
    # surface
    start = time.time()
    visfile_surface = xdmf.VisFile(directory=model_dir,
                                   domain="surface", 
                                   filename="ats_vis_surface_data.h5" , 
                                   mesh_filename="ats_vis_surface_mesh.h5")
    visfile_surface.loadMesh()
    end = time.time()
    print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
    
    # subsurface pressure (time, xy-space and soil columns)
    start = time.time()
    pressure_subsurface = visfile_subsurface.getArray('pressure')
    end = time.time()
    print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")
    
    # get the WTD (based on surface ATS ID)
    start = time.time()
    surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged = get_ats_wtd_pressurebased(pressure_subsurface=pressure_subsurface,
                                                                                    visfile_surface=visfile_surface, visfile_subsurface= visfile_subsurface)
    end = time.time()
    print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")

else:
    print("Skipping: data already extracted")

### Extract water head at the start & end points of all 2D hillslopes

- greater than 100m

In [ ]:
if flag_atsptextract:
    surface_x_coord = visfile_surface.centroids[:,0]
    surface_y_coord = visfile_surface.centroids[:,1]
    
    # Get times from surface
    surface_times = visfile_surface.times
    print(f"Number of timesteps: {len(surface_times)}")
    
    # Loop through all hillslopes
    for idx in range(num_transects):
        # Get coordinates for this hillslope
        start_coords = (start_coords_all[idx, 0], start_coords_all[idx, 1])
        end_coords = (end_coords_all[idx, 0], end_coords_all[idx, 1])
        
        # Compute Euclidean distances
        dist_start = np.sqrt((surface_x_coord - start_coords[0])**2 + (surface_y_coord - start_coords[1])**2)
        dist_end = np.sqrt((surface_x_coord - end_coords[0])**2 + (surface_y_coord - end_coords[1])**2)
        
        # Get the index of the closest point
        start_index = np.argmin(dist_start)
        end_index = np.argmin(dist_end)
        
        # Get the minimum distances
        min_dist_start = dist_start[start_index]
        min_dist_end = dist_end[end_index]
        
        # Extract head data
        startpt_head = head_rearranged[:, start_index]
        endpt_head = head_rearranged[:, end_index]
        
        # Print info
        print(f"\nHillslope {idx}: {hillslope_names[idx]}")
        print(f"  Point ID: {point_ids[idx]}, Hillslope ID: {hillslope_ids[idx]}")
        print(f"  Start point - Index: {start_index}, Distance: {min_dist_start:.2f}m")
        print(f"  End point - Index: {end_index}, Distance: {min_dist_end:.2f}m")
        
        # Create plot
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Plot start and end point heads
        ax.plot(surface_times, startpt_head, color='blue', linewidth=2, label='Start point head')
        ax.plot(surface_times, endpt_head, color='red', linewidth=2, label='End point head')
        
        ax.set_xlabel('Time [days]')
        ax.set_ylabel('Head [m]')
        ax.set_title(f'Boundary head for {hillslope_names[idx]}\n' + 
                    f'Point ID: {point_ids[idx]}, Hillslope ID: {hillslope_ids[idx]}, Distance: {distances[idx]:.1f}m')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        
        # Save plot
        plot_filename = f'./images/{site_name}/hillslopes_point{point_ids[idx]}_h{hillslope_ids[idx]}.bchead.png'
        plt.tight_layout()
        plt.savefig(plot_filename, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"  Saved plot to: {plot_filename}")
    
    print(f"\n✓ Processed all {num_transects} hillslopes")
    
else:
    print("Skipping: data already extracted")

## Save user selected hillslopes

In [ ]:
# Add this cell after the existing "## Save user selected hillslopes" section

# ==============================================================================
# USER INPUT: Select multiple hillslopes to save
# ==============================================================================
# 
# Format: List of dictionaries, each containing:
#   - 'point_id': The category point ID (from all_results)
#   - 'hillslope_id': The hillslope ID for that point
#   - 'name': A descriptive name for this transect (optional)
#
# Example:
# selected_hillslopes = [
#     {'point_id': 5, 'hillslope_id': 4, 'name': 'P5_H4'},
#     {'point_id': 17, 'hillslope_id': 5, 'name': 'P17_H5'}
# ]

selected_hillslopes = [
    {'point_id': 9, 'hillslope_id': 3, 'name': 'P9_H3'},
    {'point_id': 6, 'hillslope_id': 2, 'name': 'P6_H2'},
    {'point_id': 17, 'hillslope_id': 5, 'name': 'P17_H5'}
]

# ==============================================================================

print("\n" + "="*100)
print("SAVING SELECTED HILLSLOPE TRANSECTS")
print("="*100)

# Load the hillslopes data from the CSV created by 0a notebook
csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_all_gt100m_summary.csv'
df_all_hillslopes = pd.read_csv(csv_filename)

# Validate and collect transect data
transects_to_save = []

for idx, selection in enumerate(selected_hillslopes):
    point_id = selection['point_id']
    hillslope_id = selection['hillslope_id']
    custom_name = selection.get('name', f'P{point_id}_H{hillslope_id}')
    
    print(f"\n[{idx+1}] Processing selection: Point {point_id}, Hillslope {hillslope_id}")
    
    # Find matching row in the CSV
    matching_rows = df_all_hillslopes[
        (df_all_hillslopes['point_id'] == point_id) & 
        (df_all_hillslopes['hillslope_id'] == hillslope_id)
    ]
    
    if len(matching_rows) == 0:
        print(f"  ✗ ERROR: Point {point_id}, Hillslope {hillslope_id} not found in CSV")
        available = df_all_hillslopes[df_all_hillslopes['point_id'] == point_id]
        if len(available) > 0:
            print(f"    Available hillslopes for Point {point_id}: {available['hillslope_id'].tolist()}")
        else:
            print(f"    Point {point_id} not found. Available points: {sorted(df_all_hillslopes['point_id'].unique())}")
        continue
    
    row = matching_rows.iloc[0]
    
    # Get coordinates
    start_coords = np.array([row['start_x'], row['start_y']])
    start_z = row['start_z']
    end_coords = np.array([row['end_x'], row['end_y']])
    end_z = row['end_z']
    distance = row['distance_m']
    elev_drop = row['elev_drop_m']
    neighbor_idx = row['neighbor_idx']
    
    # Store transect data
    transects_to_save.append({
        'name': custom_name,
        'point_id': point_id,
        'hillslope_id': hillslope_id,
        'neighbor_idx': neighbor_idx,
        'start_coords': start_coords,
        'start_z': start_z,
        'end_coords': end_coords,
        'end_z': end_z,
        'distance': distance,
        'elev_drop': elev_drop
    })
    
    print(f"  ✓ Valid selection")
    print(f"    Name: {custom_name}")
    print(f"    Start: ({start_coords[0]:.2f}, {start_coords[1]:.2f}), Z={start_z:.1f}m")
    print(f"    End:   ({end_coords[0]:.2f}, {end_coords[1]:.2f}), Z={end_z:.1f}m")
    print(f"    Distance: {distance:.1f}m, Elev Drop: {elev_drop:.1f}m")
    print(f"    Neighbor Index: N{neighbor_idx}")

In [ ]:
# Save to .mat file (multi-transect format)
if len(transects_to_save) > 0:
    from scipy.io import savemat
    
    # Prepare data for MATLAB format
    start_coords_array = np.array([t['start_coords'] for t in transects_to_save])
    end_coords_array = np.array([t['end_coords'] for t in transects_to_save])
    
    # Create metadata dictionary
    metadata = {
        'names': [t['name'] for t in transects_to_save],
        'point_ids': [t['point_id'] for t in transects_to_save],
        'hillslope_ids': [t['hillslope_id'] for t in transects_to_save],
        'neighbor_indices': [t['neighbor_idx'] for t in transects_to_save],
        'distances': [t['distance'] for t in transects_to_save],
        'elev_drops': [t['elev_drop'] for t in transects_to_save],
        'start_elevations': [t['start_z'] for t in transects_to_save],
        'end_elevations': [t['end_z'] for t in transects_to_save]
    }
    
    data_to_save = {
        'start_coords': start_coords_array,
        'end_coords': end_coords_array,
        'metadata': metadata,
        'num_transects': len(transects_to_save),
        'selected_category': site_name
    }
    
    # Save to MAT file
    m2_mat_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_multi.mat'
    savemat(m2_mat_filename, data_to_save)
    
    print("\n" + "="*100)
    print("SAVE COMPLETE")
    print("="*100)
    print(f"✓ Saved {len(transects_to_save)} transect(s) to: {m2_mat_filename}")
    print(f"\nFile contents:")
    print(f"  - start_coords: [{len(transects_to_save)} x 2] array")
    print(f"  - end_coords: [{len(transects_to_save)} x 2] array")
    print(f"  - metadata: Dictionary with transect details")
    print(f"  - num_transects: {len(transects_to_save)}")
    
    print(f"\nSaved transects:")
    for i, t in enumerate(transects_to_save):
        print(f"  [{i+1}] {t['name']}: Point {t['point_id']}, Hillslope {t['hillslope_id']}, "
              f"Distance={t['distance']:.1f}m")
    
    # Also save a human-readable summary CSV
    csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_multi_summary.csv'
    summary_df = pd.DataFrame({
        'transect_name': [t['name'] for t in transects_to_save],
        'point_id': [t['point_id'] for t in transects_to_save],
        'hillslope_id': [t['hillslope_id'] for t in transects_to_save],
        'neighbor_idx': [t['neighbor_idx'] for t in transects_to_save],
        'start_x': [t['start_coords'][0] for t in transects_to_save],
        'start_y': [t['start_coords'][1] for t in transects_to_save],
        'start_z': [t['start_z'] for t in transects_to_save],
        'end_x': [t['end_coords'][0] for t in transects_to_save],
        'end_y': [t['end_coords'][1] for t in transects_to_save],
        'end_z': [t['end_z'] for t in transects_to_save],
        'distance_m': [t['distance'] for t in transects_to_save],
        'elev_drop_m': [t['elev_drop'] for t in transects_to_save]
    })
    summary_df.to_csv(csv_filename, index=False)
    print(f"\n✓ Also saved summary CSV: {csv_filename}")
    
else:
    print("\n⚠️  WARNING: No valid transects to save!")
    print("   Please check your point_id and hillslope_id selections.")

In [ ]:
# ==============================================================================
# USER INPUT: Select ONE hillslope to save as single transect
# ==============================================================================
# 
# Select the index (0-based) of the hillslope from transects_to_save
# For example: 
#   0 = first hillslope
#   1 = second hillslope
#   etc.

selected_single_index = 0  # MODIFY THIS to select which hillslope to save

# ==============================================================================

if len(transects_to_save) > 0:
    from scipy.io import savemat
    
    # Validate index
    if selected_single_index < 0 or selected_single_index >= len(transects_to_save):
        print(f"\n✗ ERROR: Invalid index {selected_single_index}")
        print(f"   Valid range: 0 to {len(transects_to_save)-1}")
        print(f"   Available transects:")
        for i, t in enumerate(transects_to_save):
            print(f"     [{i}] {t['name']}")
    else:
        # Get selected transect
        selected_transect = transects_to_save[selected_single_index]
        
        print("\n" + "="*100)
        print("SAVING SINGLE HILLSLOPE TRANSECT")
        print("="*100)
        print(f"\nSelected: [{selected_single_index}] {selected_transect['name']}")
        print(f"  Point ID: {selected_transect['point_id']}")
        print(f"  Hillslope ID: {selected_transect['hillslope_id']}")
        print(f"  Start: ({selected_transect['start_coords'][0]:.2f}, {selected_transect['start_coords'][1]:.2f}), Z={selected_transect['start_z']:.1f}m")
        print(f"  End:   ({selected_transect['end_coords'][0]:.2f}, {selected_transect['end_coords'][1]:.2f}), Z={selected_transect['end_z']:.1f}m")
        print(f"  Distance: {selected_transect['distance']:.1f}m")
        print(f"  Elev Drop: {selected_transect['elev_drop']:.1f}m")
        
        # Prepare data for single transect (compatible with original format)
        single_transect_data = {
            'start_coords': selected_transect['start_coords'],
            'end_coords': selected_transect['end_coords'],
            'metadata': {
                'name': selected_transect['name'],
                'point_id': selected_transect['point_id'],
                'hillslope_id': selected_transect['hillslope_id'],
                'neighbor_idx': selected_transect['neighbor_idx'],
                'distance': selected_transect['distance'],
                'elev_drop': selected_transect['elev_drop'],
                'start_elevation': selected_transect['start_z'],
                'end_elevation': selected_transect['end_z']
            },
            'selected_category': site_name
        }
        
        # Save to MAT file
        single_mat_filename = f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
        savemat(single_mat_filename, single_transect_data)
        
        print(f"\n✓ Saved single transect to: {single_mat_filename}")
        print(f"\nFile contents:")
        print(f"  - start_coords: [2] array")
        print(f"  - end_coords: [2] array")
        print(f"  - metadata: Dictionary with transect details")
        
        # Also save a human-readable CSV
        csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_summary.csv'
        summary_df = pd.DataFrame({
            'transect_name': [selected_transect['name']],
            'point_id': [selected_transect['point_id']],
            'hillslope_id': [selected_transect['hillslope_id']],
            'neighbor_idx': [selected_transect['neighbor_idx']],
            'start_x': [selected_transect['start_coords'][0]],
            'start_y': [selected_transect['start_coords'][1]],
            'start_z': [selected_transect['start_z']],
            'end_x': [selected_transect['end_coords'][0]],
            'end_y': [selected_transect['end_coords'][1]],
            'end_z': [selected_transect['end_z']],
            'distance_m': [selected_transect['distance']],
            'elev_drop_m': [selected_transect['elev_drop']]
        })
        summary_df.to_csv(csv_filename, index=False)
        print(f"✓ Also saved summary CSV: {csv_filename}")
        
else:
    print("\n⚠️  WARNING: No transects available to save!")
    print("   Please run the previous cell to select and save multiple transects first.")